# DINOv2 + Keras classifier

## Strategy

DINOv2 (Meta, 2023) is a Vision Transformer trained with self-supervised learning
on 142M images. Its features are among the strongest available for fine-grained
visual recognition — and artist attribution is exactly that kind of task.

**Why DINOv2 beats EfficientNetV2S / Xception for this problem:**
- Self-attention captures global style patterns across the whole canvas
- CNNs are local by design (limited receptive field per layer)
- Artist style is often a *global* property — how every part of the canvas
  relates to every other part

**Approach: offline feature extraction**
We run DINOv2 once to extract feature vectors for every image and save them
as `.npy` files. Training the Keras classifier then operates on the saved
features — no GPU memory spent re-running the transformer each epoch.

Benefits:
- Entire feature extraction runs in one pass (~minutes on GPU)
- Keras training on 768-dim vectors is extremely fast and light on VRAM
- Easy to experiment with different Keras head architectures

**Requirements:**
```
transformers
torch
torchvision
```
PyTorch is only used for feature extraction — the trained model is pure Keras.


In [1]:
import os, math
import numpy as np
from pathlib import Path
import tensorflow as tf
import keras
from keras import Model, layers
from tensorflow_addons.optimizers import AdamW
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from tensorflow_addons.metrics import F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, LearningRateScheduler


## Step 1 — Extract DINOv2 features and save to disk

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("../wikiart_split")
FEATURES_DIR  = Path("./dino_features")   # where to save .npy feature arrays
CKPT_DIR      = Path("Checkpoints")
METRICS_DIR   = Path("Metrics")
DINO_MODEL    = "facebook/dinov2-base"    # 768-dim output, ~86M params
# dinov2-large (1024-dim) gives ~1-2% more accuracy but needs more VRAM
BATCH_SIZE    = 32    # extraction batch — reduce to 16 if you hit VRAM limits
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE    = 224   # DINOv2 was pretrained at 224 — use this for extraction

print(f"Using device: {DEVICE}")
FEATURES_DIR.mkdir(exist_ok=True)


Using device: cpu


In [5]:
# ── DINOv2 preprocessing and model ───────────────────────────────────────────
processor = AutoImageProcessor.from_pretrained(DINO_MODEL)
dino      = AutoModel.from_pretrained(DINO_MODEL).to(DEVICE)
dino.eval()

print(f"DINOv2 loaded ({sum(p.numel() for p in dino.parameters()):,} params)")


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 loaded (86,580,480 params)


In [6]:
# ── Dataset helper ────────────────────────────────────────────────────────────
class ArtDataset(Dataset):
    """Walks a split directory and returns (image_tensor, label_index, image_path)."""
    IMG_EXTS = {".jpg", ".jpeg", ".png"}

    def __init__(self, split_dir: Path, class_names: list):
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples = []
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            idx = self.class_to_idx.get(class_dir.name)
            if idx is None:
                continue
            for img_path in class_dir.iterdir():
                if img_path.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((img_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        return img, label, str(path)


def collate_fn(batch):
    images, labels, paths = zip(*batch)
    inputs = processor(images=list(images), return_tensors="pt")
    return inputs, torch.tensor(labels), paths


def extract_features(split: str, class_names: list) -> tuple[np.ndarray, np.ndarray]:
    """
    Run all images in a split through DINOv2 and return (features, labels).
    Uses the [CLS] token — a single 768-dim vector per image that summarises
    the whole image, which is what we want for classification.
    """
    feat_path  = FEATURES_DIR / f"{split}_features.npy"
    label_path = FEATURES_DIR / f"{split}_labels.npy"

    if feat_path.exists() and label_path.exists():
        print(f"{split}: loading cached features from {FEATURES_DIR}")
        return np.load(feat_path), np.load(label_path)

    dataset    = ArtDataset(DATA_DIR / split, class_names)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)

    all_feats, all_labels = [], []
    print(f"Extracting {split} features ({len(dataset)} images)...")

    with torch.no_grad():
        for i, (inputs, labels, _) in enumerate(loader):
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            out    = dino(**inputs)
            cls    = out.last_hidden_state[:, 0, :]  # [CLS] token
            all_feats.append(cls.cpu().numpy())
            all_labels.append(labels.numpy())
            if (i + 1) % 20 == 0:
                print(f"  {(i+1) * BATCH_SIZE}/{len(dataset)}")

    feats  = np.concatenate(all_feats,  axis=0).astype(np.float32)
    labels = np.concatenate(all_labels, axis=0).astype(np.int32)

    np.save(feat_path,  feats)
    np.save(label_path, labels)
    print(f"  Saved to {feat_path}  shape={feats.shape}")
    return feats, labels


In [7]:
# ── Run extraction for all three splits ──────────────────────────────────────
# Uses class_names from the training split as the canonical ordering.
_tmp_ds    = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", batch_size=None, image_size=(64, 64))
class_names = _tmp_ds.class_names
N_CLASSES   = len(class_names)
print(f"Classes ({N_CLASSES}): {class_names}")
del _tmp_ds

train_feats, train_labels = extract_features("train", class_names)
val_feats,   val_labels   = extract_features("val",   class_names)
test_feats,  test_labels  = extract_features("test",  class_names)

print(f"\nFeature shapes — train: {train_feats.shape}, val: {val_feats.shape}, test: {test_feats.shape}")
print(f"Feature dim: {train_feats.shape[1]}")  # 768 for dinov2-base


Found 9326 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']
Extracting train features (9326 images)...
  640/9326
  1280/9326
  1920/9326
  2560/9326
  3200/9326
  3840/9326
  4480/9326
  5120/9326
  5760/9326
  6400/9326
  7040/9326
  7680/9326
  8320/9326
  8960/9326
  Saved to dino_features\train_features.npy  shape=(9326, 768)
Extracting val features (1992 images)...
  640/1992
  1280/1992
  1920/1992
  Saved to dino_features\val_features.npy  shape=(1992, 768)
Extracting test features (2022 images)...
  640/2022
  1280/2022
  1920/2022
  Saved to dino_features\test_features.n

## Step 2 — Build and train the Keras classifier

The Keras model receives 768-dim DINOv2 feature vectors as input.
This is the "model implementation in Keras" for the purposes of the project.


In [8]:
# ── Keras tf.data pipelines from numpy arrays ─────────────────────────────────
KERAS_BATCH = 256   # features are tiny vectors — large batches are fast and stable
AUTOTUNE    = tf.data.AUTOTUNE

def to_onehot(labels, n_classes):
    return tf.one_hot(labels, n_classes).numpy()

train_labels_oh = to_onehot(train_labels, N_CLASSES)
val_labels_oh   = to_onehot(val_labels,   N_CLASSES)
test_labels_oh  = to_onehot(test_labels,  N_CLASSES)

train_tf = (tf.data.Dataset.from_tensor_slices((train_feats, train_labels_oh))
              .shuffle(len(train_feats), seed=123)
              .batch(KERAS_BATCH)
              .prefetch(AUTOTUNE))
val_tf   = (tf.data.Dataset.from_tensor_slices((val_feats, val_labels_oh))
              .batch(KERAS_BATCH).prefetch(AUTOTUNE))
test_tf  = (tf.data.Dataset.from_tensor_slices((test_feats, test_labels_oh))
              .batch(KERAS_BATCH).prefetch(AUTOTUNE))


In [9]:
# ── Keras classification head ─────────────────────────────────────────────────
# A relatively deep MLP is appropriate here: DINOv2 features are rich but
# require non-linear transformation to produce good class boundaries.
# We also apply L2 regularisation on the Dense weights (weight_decay on the
# optimizer is less effective for a shallow MLP than for a full CNN).

from keras.regularizers import l2

def build_dino_classifier(input_dim=768, n_classes=23, dropout=0.4, l2_reg=1e-4):
    inp = keras.Input(shape=(input_dim,), name="dino_features")

    # Normalise the incoming features — DINOv2 CLS tokens can vary in scale
    x = layers.LayerNormalization(name="input_norm")(inp)

    x = layers.Dense(1024, kernel_regularizer=l2(l2_reg), name="fc1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("gelu", name="act1")(x)     # GELU matches DINOv2 internals
    x = layers.Dropout(dropout, name="drop1")(x)

    x = layers.Dense(512, kernel_regularizer=l2(l2_reg), name="fc2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Activation("gelu", name="act2")(x)
    x = layers.Dropout(dropout, name="drop2")(x)

    x = layers.Dense(256, kernel_regularizer=l2(l2_reg), name="fc3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    x = layers.Activation("gelu", name="act3")(x)
    x = layers.Dropout(dropout / 2, name="drop3")(x)

    out = layers.Dense(n_classes, activation="softmax", dtype="float32", name="head")(x)
    return Model(inp, out, name="dino_classifier")

model_dino = build_dino_classifier(
    input_dim=train_feats.shape[1],
    n_classes=N_CLASSES,
)
model_dino.summary()


Model: "dino_classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dino_features (InputLayer)  [(None, 768)]             0         
                                                                 
 input_norm (LayerNormalizat  (None, 768)              1536      
 ion)                                                            
                                                                 
 fc1 (Dense)                 (None, 1024)              787456    
                                                                 
 bn1 (BatchNormalization)    (None, 1024)              4096      
                                                                 
 act1 (Activation)           (None, 1024)              0         
                                                                 
 drop1 (Dropout)             (None, 1024)              0         
                                                   

In [ ]:
# ── LR schedule ───────────────────────────────────────────────────────────────
DINO_EPOCHS = 60
DINO_LR     = 3e-4   # AdamW default; slightly higher than CNN fine-tuning
              # because we are training from scratch on a fixed feature space

def cosine_warmup(base_lr, total_epochs, warmup_epochs=5):
    def sched(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * p))
    return sched

model_dino.compile(
    optimizer=AdamW(learning_rate=DINO_LR, weight_decay=1e-6),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        F1Score(average="macro", name="f1_score", num_classes=N_CLASSES),
    ],
)

callbacks = [
    ModelCheckpoint( CKPT_DIR / "ckpt_dino.keras", monitor="val_loss",
                    save_best_only=True, verbose=1),
    CSVLogger(METRICS_DIR / "log_dino.csv"),
    LearningRateScheduler(cosine_warmup(DINO_LR, DINO_EPOCHS, warmup_epochs=5)),
    EarlyStopping(monitor="val_loss", patience=10,
                  restore_best_weights=True, verbose=1),
]

history = model_dino.fit(
    train_tf,
    validation_data=val_tf,
    epochs=DINO_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/60
35/37 [===========================>..] - ETA: 0s - loss: 3.5396 - accuracy: 0.0844 - auc: 0.5819 - f1_score: 0.0693
Epoch 1: val_loss improved from inf to 2.99893, saving model to ckpt_dino.keras
37/37 [==============================] - 6s 36ms/step - loss: 3.5285 - accuracy: 0.0873 - auc: 0.5847 - f1_score: 0.0712 - val_loss: 2.9989 - val_accuracy: 0.2987 - val_auc: 0.7707 - val_f1_score: 0.2558 - lr: 6.0000e-05
Epoch 2/60
35/37 [===========================>..] - ETA: 0s - loss: 2.8671 - accuracy: 0.2723 - auc: 0.7630 - f1_score: 0.2260
Epoch 2: val_loss improved from 2.99893 to 2.44059, saving model to ckpt_dino.keras
37/37 [==============================] - 1s 30ms/step - loss: 2.8573 - accuracy: 0.2765 - auc: 0.7652 - f1_score: 0.2300 - val_loss: 2.4406 - val_accuracy: 0.4558 - val_auc: 0.8855 - val_f1_score: 0.3909 - lr: 1.2000e-04
Epoch 3/60
37/37 [==============================] - ETA: 0s - loss: 2.3909 - accuracy: 0.4372 - auc: 0.8683 - f1_score: 0.3802
Epoch 3: val_

In [12]:
# ── Final evaluation ──────────────────────────────────────────────────────────
results = model_dino.evaluate(test_tf, return_dict=True)
print(f"\nDINOv2 classifier — test results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")


8/8 [==============================] - 0s 9ms/step - loss: 1.3672 - accuracy: 0.8111 - auc: 0.9821 - f1_score: 0.8014

DINOv2 classifier — test results:
  loss: 1.3672
  accuracy: 0.8111
  auc: 0.9821
  f1_score: 0.8014
